In [1]:
!pip install fitparse gpxpy chromadb langchain-community sentence-transformers

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
   ---------------------------------------- 0.0/23.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/23.5 MB ? eta -:--:--
   ---------------------------------------- 0.3/23.5 MB ? eta -:--:--
   - -------------------------------------- 0.8/23.5 MB 1.4 MB/s eta 0:00:17
   - -------------------------------------- 1.0/23.5 MB 1.5 MB/s eta 0:00:16
   -- ------------------------------------- 1.6/23.5 MB 1.7 MB/s eta 0:00:14
   --- ------------------------------------ 2.1/23.5 MB 1.8 MB/s eta 0:00:12
   ---- ----------------------------------- 2.9/23.5 MB 2.1 MB/s eta 0:00:10
   ------ -


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import requests

try:
  response = requests.get('http://localhost:11434/api/tags')
  if response.status_code == 200:
    print('Ollama background service is running!')
    print('Installed models:', response.json())
except Exception as e:
  print('Could not connect to Ollama server:', e)

Ollama background service is running!
Installed models: {'models': [{'name': 'llama3:latest', 'model': 'llama3:latest', 'modified_at': '2026-09-29T12:57:48.0084871+02:00', 'size': 4661224676, 'digest': '365c0bd3c000a25d28ddbf732fe1c6add414de7275464c4e4d1c3b5fcb5d8ad1', 'details': {'parent_model': '', 'format': 'gguf', 'family': 'llama', 'families': ['llama'], 'parameter_size': '8.0B', 'quantization_level': 'Q4_0', 'context_length': 8192, 'embedding_length': 4096}, 'capabilities': ['completion']}]}


In [ ]:
import pandas as pd
import requests
import re

# Load your original Strava CSV directly
df = pd.read_csv("activities.csv")
df['Afstand'] = (
  df['Afstand']
  .astype(str)
  .str.replace('.', '', regex=False)  # Remove thousands separator dot
  .str.replace(',', '.', regex=False)  # Convert decimal comma to dot
  .astype('float64')
)

column_mapping = {
    'Activiteits-ID': 'Activity ID',
    'Datum van activiteit': 'Date',
    'Naam activiteit': 'Activity Name',
    'Activiteitstype': 'Activity Type',
    'Beschrijving van activiteit': 'Description',
    'Fiets': 'Bike Used',
    'Uitrusting voor activiteit': 'Gear Used',
    'Bestandsnaam': 'Filename',
    'Media': 'Media Count',
    'Beweegtijd': 'Moving Time',
    'Verstreken tijd': 'Elapsed Time',
    'Verstreken tijd.1': 'Elapsed Time (Detailed)',
    'Afstand': 'Distance (km)',
    'Afstand.1': 'Distance (meters)',
    'Afstand (onverharde wegen)': 'Unpaved Distance (m)',
    'Gemiddelde snelheid': 'Average Speed',
    'Gemiddelde snelheid (op basis van verstreken tijd)': 'Average Speed (Elapsed)',
    'Max. snelheid': 'Max Speed',
    'Totale stijging': 'Elevation Gain',
    'Totale daling': 'Elevation Loss',
    'Kleinste hoogte': 'Min Elevation',
    'Grootste hoogte': 'Max Elevation',
    'Gemiddeld stijgingspercentage': 'Average Grade',
    'Max. stijgingspercentage': 'Max Grade',
    'Gemiddeld wattage': 'Average Power (W)',
    'Gewogen gemiddeld vermogen': 'Weighted Avg Power / NP (W)',
    'Totale arbeid': 'Total Work (kJ)',
    'Aantal vermogensgegevens': 'Power Samples Count',
    'Gemiddelde hartslag': 'Average HR',
    'Max. hartslag': 'Max HR',
    'Max. hartslag.1': 'Max HR (Peak)',
    'Gemiddelde cadans': 'Average Cadence',
    'Max. cadans': 'Max Cadence',
    'Calorieën': 'Calories',
    'Vergelijkbare poging': 'Relative Effort',
    'Vergelijkbare poging.1': 'Relative Effort (Detailed)',
    'Ervaren vergelijkbare poging': 'Perceived Relative Effort',
    'Woon-werkverkeer': 'Commute',
    'Buitentemperatuur': 'Outside Temperature',
    'Gevoelstemperatuur': 'Apparent Temperature',
    'Maximale temperatuur': 'Max Temperature',
    'Gemiddelde temperatuur': 'Average Temperature',
    'Tijd weerbeeld': 'Weather Timestamp',
    'Weersomstandigheden': 'Weather Summary',
    'Dauwpunt': 'Dew Point',
    'Vochtigheid': 'Humidity',
    'Luchtdruk': 'Air Pressure',
    'Windsnelheid': 'Wind Speed',
    'Windstoot': 'Wind Gust',
    'Windrichting': 'Wind Direction',
    'Bewolking': 'Cloud Cover',
    'Zicht': 'Visibility',
    'UV-index': 'UV Index',
    'Kans op neerslag': 'Precipitation Probability',
    'Neerslagintensiteit': 'Precipitation Intensity',
    'Type neerslag': 'Precipitation Type',
    'Tijd zonsopgang': 'Sunrise Time',
    'Tijd zonsondergang': 'Sunset Time',
}

df.rename(columns=column_mapping, inplace=True)

# Ensure Activity ID is a string so it can serve as ChromaDB primary key
df['Activity ID'] = df['Activity ID'].astype(str)

def ask_table_llm(user_query):
  # 1. Feed the LLM only the column structure and data types, not the full table
 
  schema_info = df.dtypes.to_string()

  system_prompt = f"""You are a pandas Python code generator. 
    DataFrame `df` schema (Dutch headers):
    {schema_info}

    Key Column Mappings:
    - 'date' / 'when' -> 'Datum' or 'Datum van activiteit'
    - 'distance' -> 'Afstand' (in kilometers, e.g. 50 means 50km)
    - 'wattage' / 'power' -> 'Gemiddeld wattage'

    Rules:
    1. Return ONLY 1 line of executable Python code that evaluates directly to the answer.
    2. For questions asking 'which date/ride/item' satisfies a max/min condition, extract the specific value (e.g. `df[df['Distance (km)'] > 50].sort_values('Average Power (W)', ascending=False).iloc[0]['Date']`).
    3. NEVER return `.idxmax()` or index integers alone.
    4. Output raw code ONLY. No markdown, no triple backticks."""

  # 2. Ask Llama 3 to write the code
  response = requests.post(
      "http://localhost:11434/api/generate",
      json={
          "model": "qwen2.5-coder:3b",
          "prompt": f"{system_prompt}\n\nQuestion: {user_query}\nCode:",
          "stream": False,
          "options": {
              "temperature": 0.0,
              "num_predict": 100  # Give it enough room to generate
          },
      },
  )

  data = response.json()
  if "error" in data:
      return f"Ollama Error: {data['error']}"

  raw_code = data.get("response", "").strip()

  # Clean markdown formatting backticks if the LLM still includes them
  clean_code = re.sub(r"```(?:python)?|```", "", raw_code).strip()
  clean_code = clean_code.split("\n")[0].strip()  # Take only the first line

  # Execute on DataFrame
  try:
      result = eval(clean_code)
      return f"Query: {clean_code}\nResult: {result}"
  except Exception as e:
      return f"Failed to execute `{clean_code}`: {e}\n(Raw output was: `{raw_code}`)"

# Example Usage:
print(ask_table_llm("What was my highest average wattage on a ride over 50km?"))
print(ask_table_llm("On which date was the ride over 50km with the highest average wattage?"))

Query: df[df['Distance (km)'] > 50]['Average Power (W)'].max()
Result: 290.0
Query: df[df['Distance (km)'] > 50].sort_values('Average Power (W)', ascending=False).iloc[0]['Date']
Result: 4 feb 2022, 15:00:48


In [53]:
import chromadb
from chromadb.utils import embedding_functions
import pandas as pd
import requests

DB_PATH = "./chroma_db"
CSV_PATH = "activities.csv"

# 1. Initialize Persistent Client (Saves embeddings to disk)
chroma_client = chromadb.PersistentClient(path=DB_PATH)

# try:
#     chroma_client.delete_collection(name="strava_text_search")
#     print("Deleted stale collection. Building fresh index...")
# except Exception:
#     pass

collection = chroma_client.get_or_create_collection(name="strava_text_search")
df['Activity ID'] = df['Activity ID'].astype(str).str.strip()

# 2. Only process and index CSV if the collection is empty
if collection.count() == 0:
    print("Indexing activities into ChromaDB for the first time...")

    # df is defined in the previous cell, so we can use it directly
    
    chroma_client = chromadb.PersistentClient(path=DB_PATH)

    try:
        chroma_client.delete_collection(name="strava_text_search")
    except Exception:
        pass

    collection = chroma_client.get_or_create_collection(name="strava_text_search")

    documents, metadatas, ids = [], [], []

    for idx, row in df.iterrows():
        act_id = str(row['Activity ID']).strip()
        
        # Safely convert to clean string
        name = "" if pd.isna(row.get('Activity Name')) else str(row.get('Activity Name')).strip()
        desc = "" if pd.isna(row.get('Description')) else str(row.get('Description')).strip()
        bike = "" if pd.isna(row.get('Bike Used')) else str(row.get('Bike Used')).strip()
        act_type = "" if pd.isna(row.get('Activity Type')) else str(row.get('Activity Type')).strip()

        parts = []
        if name: parts.append(f"Title: {name}")
        if act_type: parts.append(f"Type: {act_type}")
        if bike: parts.append(f"Bike: {bike}")
        if desc: parts.append(f"Description: {desc}")

        combined_text = " | ".join(parts) if parts else "Unspecified activity"

        documents.append(combined_text)
        ids.append(act_id)
        metadatas.append({"activity_id": act_id, "date": str(row.get('Date', '')), "name": name})

    collection.upsert(documents=documents, metadatas=metadatas, ids=ids)
    print(f"\nIndexed {collection.count()} rides to ChromaDB.")



def search_rides(user_query, top_k=5):
    query_str = user_query.strip()
    all_data = collection.get(include=["documents"])
    
    # 1. Exact Substring Search (Fast, 100% accurate for single words/names)
    matched_ids = [
        doc_id for doc_id, doc_text in zip(all_data["ids"], all_data["documents"])
        if query_str.lower() in doc_text.lower()
    ]
    
    if matched_ids:
        print(f"--- Direct Text Match for '{query_str}' ({len(matched_ids)} ride(s) found) ---")
        return df[df['Activity ID'].isin(matched_ids[:top_k])]

    # 2. Semantic Vector Fallback (For broad natural language like "mud in village X")
    print(f"--- Semantic Vector Search for '{query_str}' ---")
    results = collection.query(query_texts=[query_str], n_results=top_k)
    if results and results['ids'] and len(results['ids'][0]) > 0:
        matched_ids = [str(x) for x in results['ids'][0]]
        return df[df['Activity ID'].isin(matched_ids)]
    
    return pd.DataFrame()

# Tests:
# Single word keyword match -> Returns Row 764 & Row 2
print(search_rides("Ventoux"))

# Natural language semantic query -> Works via vector embeddings
print(search_rides("Ride over the Ventoux"))

--- Direct Text Match for 'Ventoux' (2 ride(s) found) ---
     Activity ID                   Date                         Activity Name  \
2    20308489377  24 sep 2026, 06:46:27                            Sveti Jure   
764  12478082074  22 sep 2024, 00:47:32  Ventoux challenge teambuilding event   

    Activity Type                                        Description  \
2        Fietsrit  The Mont Ventoux of the Dinaric Alps. With a g...   
764      Fietsrit                                 6 times to the top   

     Verstreken tijd Distance (km)  Max. hartslag  Vergelijkbare poging  \
2              13819         69,67          168.0                 313.0   
764            56814        274,23          163.0                 985.0   

     Woon-werkverkeer  ...  Herstel Met huisdier Wedstrijd  Lange loop  \
2               False  ...      NaN          NaN       NaN         NaN   
764             False  ...      NaN          NaN       NaN         NaN   

     Voor een doel  Met kind  Af

In [ ]:
import chromadb
from chromadb.utils import embedding_functions
import pandas as pd

DB_PATH = "./chroma_db"

# 1. Initialize Persistent Client
chroma_client = chromadb.PersistentClient(path=DB_PATH)

# Ensure fresh collection structure during setup
collection = chroma_client.get_or_create_collection(name="strava_text_search")

# Ensure Activity ID is string typed in pandas
df['Activity ID'] = df['Activity ID'].astype(str).str.strip()

# 2. Index CSV
if collection.count() == 0:
    print("Indexing activities into ChromaDB...")

    documents, metadatas, ids = [], [], []

    for idx, row in df.iterrows():
        act_id = str(row['Activity ID']).strip()
        
        name = "" if pd.isna(row.get('Activity Name')) else str(row.get('Activity Name')).strip()
        desc = "" if pd.isna(row.get('Description')) else str(row.get('Description')).strip()
        bike = "" if pd.isna(row.get('Bike Used')) else str(row.get('Bike Used')).strip()
        act_type = "" if pd.isna(row.get('Activity Type')) else str(row.get('Activity Type')).strip()

        # Format as natural sentences instead of pipe-delimited key-values
        sentences = []
        if name: 
            sentences.append(name if name.endswith('.') else f"{name}.")
        if act_type: 
            sentences.append(f"Activity type is {act_type}.")
        if bike: 
            sentences.append(f"Ridden on {bike}.")
        if desc: 
            sentences.append(desc)

        combined_text = " ".join(sentences) if sentences else "Unspecified activity"

        documents.append(combined_text)
        ids.append(act_id)
        metadatas.append({"activity_id": act_id, "date": str(row.get('Date', '')), "name": name})

    collection.upsert(documents=documents, metadatas=metadatas, ids=ids)
    print(f"Indexed {collection.count()} rides to ChromaDB.")


def search_rides(user_query, top_k=5, max_distance=1.2):
    query_str = user_query.strip()
    all_data = collection.get(include=["documents"])
    
    # 1. Exact Substring Search
    matched_ids = [
        doc_id for doc_id, doc_text in zip(all_data["ids"], all_data["documents"])
        if query_str.lower() in doc_text.lower()
    ]
    
    if matched_ids:
        print(f"--- Direct Text Match for '{query_str}' ({len(matched_ids)} match(es)) ---")
        matched_subset = matched_ids[:top_k]
        result_df = df[df['Activity ID'].isin(matched_subset)].copy()
        # Preserve matching rank order
        result_df['sort_key'] = result_df['Activity ID'].map(lambda x: matched_subset.index(x))
        return result_df.sort_values('sort_key').drop(columns=['sort_key'])

    # 2. Semantic Vector Fallback
    print(f"--- Semantic Vector Search for '{query_str}' ---")
    results = collection.query(
        query_texts=[query_str], 
        n_results=top_k,
        include=["documents", "distances"]
    )
    
    if results and results['ids'] and len(results['ids'][0]) > 0:
        returned_ids = [str(x) for x in results['ids'][0]]
        distances = results['distances'][0]
        
        # Filter out low-relevance results using a distance threshold
        filtered_ids = [
            doc_id for doc_id, dist in zip(returned_ids, distances) 
            if dist <= max_distance
        ]
        
        if not filtered_ids:
            print("No vector results met the minimum similarity threshold.")
            return pd.DataFrame()
            
        print(f"Top result distance: {distances[0]:.4f}")
        
        # Filter DataFrame and maintain vector similarity ranking
        result_df = df[df['Activity ID'].isin(filtered_ids)].copy()
        result_df['sort_key'] = result_df['Activity ID'].map(lambda x: filtered_ids.index(x))
        return result_df.sort_values('sort_key').drop(columns=['sort_key'])
    
    return pd.DataFrame()

# Run tests
print(search_rides("Ventoux"))
print(search_rides("Ride over the Ventoux"))

--- Direct Text Match for 'Ventoux' (2 match(es)) ---
     Activity ID                   Date                         Activity Name  \
2    20308489377  24 sep 2026, 06:46:27                            Sveti Jure   
764  12478082074  22 sep 2024, 00:47:32  Ventoux challenge teambuilding event   

    Activity Type                                        Description  \
2        Fietsrit  The Mont Ventoux of the Dinaric Alps. With a g...   
764      Fietsrit                                 6 times to the top   

     Verstreken tijd Distance (km)  Max. hartslag  Vergelijkbare poging  \
2              13819         69,67          168.0                 313.0   
764            56814        274,23          163.0                 985.0   

     Woon-werkverkeer  ...  Herstel Met huisdier Wedstrijd  Lange loop  \
2               False  ...      NaN          NaN       NaN         NaN   
764             False  ...      NaN          NaN       NaN         NaN   

     Voor een doel  Met kind  Afstan

In [49]:
import pandas as pd
import chromadb

DB_PATH = "./chroma_db"
CSV_PATH = "activities.csv"

# 1. Inspect Raw Pandas DataFrame First
df = pd.read_csv(CSV_PATH)

# Clean column headers
column_mapping = {
    'Activiteits-ID': 'Activity ID',
    'Datum van activiteit': 'Date',
    'Naam activiteit': 'Activity Name',
    'Beschrijving van activiteit': 'Description',
    'Activiteitstype': 'Activity Type',
    'Fiets': 'Bike Used',
    'Afstand': 'Distance (km)',
}
df.rename(columns=column_mapping, inplace=True)

# Find Ventoux row in raw CSV to verify it exists
raw_ventoux = df[df['Activity Name'].astype(str).str.contains('Ventoux', case=False, na=False)]

print("=== RAW CSV CHECK ===")
if raw_ventoux.empty:
    print("CRITICAL: 'Ventoux' was NOT found in the 'Activity Name' column of your CSV!")
else:
    print(f"Found {len(raw_ventoux)} row(s) containing 'Ventoux' in CSV:")
    for idx, r in raw_ventoux.iterrows():
        print(f"  ID: {r.get('Activity ID')} | Title: '{r.get('Activity Name')}'")

# 2. Re-build ChromaDB Collection safely
chroma_client = chromadb.PersistentClient(path=DB_PATH)

try:
    chroma_client.delete_collection(name="strava_text_search")
except Exception:
    pass

collection = chroma_client.get_or_create_collection(name="strava_text_search")

documents, metadatas, ids = [], [], []

for idx, row in df.iterrows():
    act_id = str(row['Activity ID']).strip()
    
    # Safely convert to clean string
    name = "" if pd.isna(row.get('Activity Name')) else str(row.get('Activity Name')).strip()
    desc = "" if pd.isna(row.get('Description')) else str(row.get('Description')).strip()
    bike = "" if pd.isna(row.get('Bike Used')) else str(row.get('Bike Used')).strip()
    act_type = "" if pd.isna(row.get('Activity Type')) else str(row.get('Activity Type')).strip()

    parts = []
    if name: parts.append(f"Title: {name}")
    if act_type: parts.append(f"Type: {act_type}")
    if bike: parts.append(f"Bike: {bike}")
    if desc: parts.append(f"Description: {desc}")

    combined_text = " | ".join(parts) if parts else "Unspecified activity"

    documents.append(combined_text)
    ids.append(act_id)
    metadatas.append({"activity_id": act_id, "date": str(row.get('Date', '')), "name": name})

collection.upsert(documents=documents, metadatas=metadatas, ids=ids)
print(f"\nIndexed {collection.count()} rides to ChromaDB.")

# 3. Test Search
def find_rides(user_query, top_k=5):
    query_str = user_query.strip()
    
    # Fetch all stored documents from ChromaDB
    all_data = collection.get(include=["documents"])
    
    matched_ids = [
        doc_id for doc_id, doc_text in zip(all_data["ids"], all_data["documents"])
        if query_str.lower() in doc_text.lower()
    ]
    
    if matched_ids:
        print(f"\n=== SUCCESS: Found {len(matched_ids)} match(es) for '{query_str}' ===")
        return df[df['Activity ID'].astype(str).isin(matched_ids[:top_k])]
    else:
        print(f"\n=== FAIL: '{query_str}' still not matched in stored documents ===")
        return pd.DataFrame()

# Test
print(find_rides("Ventoux"))

=== RAW CSV CHECK ===
Found 1 row(s) containing 'Ventoux' in CSV:
  ID: 12478082074 | Title: 'Ventoux challenge teambuilding event'

Indexed 2307 rides to ChromaDB.

=== SUCCESS: Found 2 match(es) for 'Ventoux' ===
     Activity ID                   Date                         Activity Name  \
2    20308489377  24 sep 2026, 06:46:27                            Sveti Jure   
764  12478082074  22 sep 2024, 00:47:32  Ventoux challenge teambuilding event   

    Activity Type                                        Description  \
2        Fietsrit  The Mont Ventoux of the Dinaric Alps. With a g...   
764      Fietsrit                                 6 times to the top   

     Verstreken tijd Distance (km)  Max. hartslag  Vergelijkbare poging  \
2              13819         69,67          168.0                 313.0   
764            56814        274,23          163.0                 985.0   

     Woon-werkverkeer  ...  Herstel Met huisdier Wedstrijd  Lange loop  \
2               False  .

In [ ]:
# Load your Strava export CSV
df = pd.read_csv("activities.csv")

# Generate text documents for embedding
documents = [build_activity_text(row) for _, row in df.iterrows()]

# Print preview of the first document
print(documents[0])

['=== ACTIVITY LOG: Afternoon Ride (28 sep 2026, 11:04:44) ===\nActivity ID: 20363851210\nDate: 28 sep 2026, 11:04:44\nActivity Name: Afternoon Ride\nActivity Type: Fietsrit\nBike Used: 12241696.0\nGear Used: Jeagher\nFilename: activities/21519095837.fit.gz\nMedia Count: |\nMoving Time: 6738.0\nElapsed Time: 7019\nElapsed Time (Detailed): 7019.0\nDistance (km): 54,53\nDistance (meters): 54530.3\nUnpaved Distance (m): 151.1\nAverage Speed: 8.093\nAverage Speed (Elapsed): 7.769\nMax Speed: 16.14\nElevation Gain: 666.0\nElevation Loss: 661.0\nMin Elevation: 190.8\nMax Elevation: 426.4\nAverage Grade: 0.0\nMax Grade: 49.8\nAverage Power (W): 163.0\nAverage HR: 127.0\nMax HR: 164.0\nMax HR (Peak): 164.0\nCalories: 1259.0\nRelative Effort: 108.0\nRelative Effort (Detailed): 108.0\nCommute: No\nOutside Temperature: 21.8\nApparent Temperature: 22.2\nAverage Temperature: 24.6\nWeather Timestamp: 1790593200.0\nWeather Summary: 1.0\nDew Point: 9.0\nHumidity: 0.44\nAir Pressure: 1025.4\nWind Speed

In [7]:
max(len(doc) for doc in documents)

9201

--- Direct Text Match for 'Ventoux' (2 match(es)) ---
     Activity ID                   Date                         Activity Name  \
2    20308489377  24 sep 2026, 06:46:27                            Sveti Jure   
764  12478082074  22 sep 2024, 00:47:32  Ventoux challenge teambuilding event   

    Activity Type                                        Description  \
2        Fietsrit  The Mont Ventoux of the Dinaric Alps. With a g...   
764      Fietsrit                                 6 times to the top   

     Verstreken tijd Distance (km)  Max. hartslag  Vergelijkbare poging  \
2              13819         69,67          168.0                 313.0   
764            56814        274,23          163.0                 985.0   

     Woon-werkverkeer  ...  Herstel Met huisdier Wedstrijd  Lange loop  \
2               False  ...      NaN          NaN       NaN         NaN   
764             False  ...      NaN          NaN       NaN         NaN   

     Voor een doel  Met kind  Afstan

In [60]:
import chromadb
from chromadb.utils import embedding_functions
import pandas as pd

DB_PATH = "./chroma_db"

# 1. Initialize Persistent Client
chroma_client = chromadb.PersistentClient(path=DB_PATH)

# Ensure fresh collection structure during setup
collection = chroma_client.get_or_create_collection(name="strava_text_search")

# Ensure Activity ID is string typed in pandas
df['Activity ID'] = df['Activity ID'].astype(str).str.strip()

# 2. Index CSV
if collection.count() == 0:
    print("Indexing activities into ChromaDB...")

    documents, metadatas, ids = [], [], []

    for idx, row in df.iterrows():
        act_id = str(row['Activity ID']).strip()
        
        name = "" if pd.isna(row.get('Activity Name')) else str(row.get('Activity Name')).strip()
        desc = "" if pd.isna(row.get('Description')) else str(row.get('Description')).strip()
        bike = "" if pd.isna(row.get('Bike Used')) else str(row.get('Bike Used')).strip()
        act_type = "" if pd.isna(row.get('Activity Type')) else str(row.get('Activity Type')).strip()

        # Format as natural sentences instead of pipe-delimited key-values
        sentences = []
        if name: 
            sentences.append(name if name.endswith('.') else f"{name}.")
        if act_type: 
            sentences.append(f"Activity type is {act_type}.")
        if bike: 
            sentences.append(f"Ridden on {bike}.")
        if desc: 
            sentences.append(desc)

        combined_text = " ".join(sentences) if sentences else "Unspecified activity"

        documents.append(combined_text)
        ids.append(act_id)
        metadatas.append({"activity_id": act_id, "date": str(row.get('Date', '')), "name": name})

    collection.upsert(documents=documents, metadatas=metadatas, ids=ids)
    print(f"Indexed {collection.count()} rides to ChromaDB.")


def search_rides(user_query, top_k=5, max_distance=1.2):
    query_str = user_query.strip()
    all_data = collection.get(include=["documents"])
    
    # 1. Exact Substring Search
    matched_ids = [
        doc_id for doc_id, doc_text in zip(all_data["ids"], all_data["documents"])
        if query_str.lower() in doc_text.lower()
    ]
    
    if matched_ids:
        print(f"--- Direct Text Match for '{query_str}' ({len(matched_ids)} match(es)) ---")
        matched_subset = matched_ids[:top_k]
        result_df = df[df['Activity ID'].isin(matched_subset)].copy()
        # Preserve matching rank order
        result_df['sort_key'] = result_df['Activity ID'].map(lambda x: matched_subset.index(x))
        return result_df.sort_values('sort_key').drop(columns=['sort_key'])

    # 2. Semantic Vector Fallback
    print(f"--- Semantic Vector Search for '{query_str}' ---")
    results = collection.query(
        query_texts=[query_str], 
        n_results=top_k,
        include=["documents", "distances"]
    )
    
    if results and results['ids'] and len(results['ids'][0]) > 0:
        returned_ids = [str(x) for x in results['ids'][0]]
        distances = results['distances'][0]
        
        # Filter out low-relevance results using a distance threshold
        filtered_ids = [
            doc_id for doc_id, dist in zip(returned_ids, distances) 
            if dist <= max_distance
        ]
        
        if not filtered_ids:
            print("No vector results met the minimum similarity threshold.")
            return pd.DataFrame()
            
        print(f"Top result distance: {distances[0]:.4f}")
        
        # Filter DataFrame and maintain vector similarity ranking
        result_df = df[df['Activity ID'].isin(filtered_ids)].copy()
        result_df['sort_key'] = result_df['Activity ID'].map(lambda x: filtered_ids.index(x))
        return result_df.sort_values('sort_key').drop(columns=['sort_key'])
    
    return pd.DataFrame()

# Run tests
print(search_rides("Tourmalet"))
print(search_rides("Ride over the Tourmalet"))


--- Direct Text Match for 'Tourmalet' (2 match(es)) ---
     Activity ID                   Date  \
393  16082376772   7 okt 2025, 22:27:45   
769  12424582339  16 sep 2024, 13:14:44   

                                         Activity Name      Activity Type  \
393                           TransPyrenees Race - 6/7           Fietsrit   
769  ROUVY - Col du Tourmalet from Luz Saint Sauveu...  Virtuele fietsrit   

                                           Description  Verstreken tijd  \
393  I got up after a bit longer sleep than usual a...            80215   
769                                                NaN             4258   

    Distance (km)  Max. hartslag  Vergelijkbare poging  Woon-werkverkeer  ...  \
393        304,29          132.0                 263.0             False  ...   
769         19,21          154.0                 110.0             False  ...   

     Herstel Met huisdier Wedstrijd  Lange loop  Voor een doel  Met kind  \
393      0.0          NaN       NaN 

In [61]:
import pandas as pd
import chromadb
import re

DB_PATH = "./chroma_db"
chroma_client = chromadb.PersistentClient(path=DB_PATH)
collection = chroma_client.get_or_create_collection(name="strava_text_search")

# Common English stop words in activity searches
STOP_WORDS = {
    "ride", "riding", "over", "the", "a", "an", "in", "on", "to", "of", "and", 
    "with", "up", "down", "for", "my", "by", "from", "trip", "climb", "cycling"
}

def search_rides(user_query, top_k=5, max_distance=1.25):
    raw_query = user_query.strip().lower()
    
    # Extract meaningful search terms (ignore stop words and tiny words <= 2 chars)
    tokens = [
        re.sub(r'[^a-z0-9]', '', w) 
        for w in raw_query.split() 
        if w not in STOP_WORDS and len(w) > 2
    ]
    
    all_data = collection.get(include=["documents"])
    all_ids = all_data["ids"]
    all_docs = all_data["documents"]
    
    # 1. TOKEN-BASED KEYWORD MATCHING
    # Checks if any key entity (e.g. "tourmalet", "ventoux") exists anywhere in the text
    matched_ids = []
    if tokens:
        for doc_id, doc_text in zip(all_ids, all_docs):
            text_lower = doc_text.lower()
            # If ALL key tokens (or ANY key token if only 1 exists) match the document
            if all(token in text_lower for token in tokens):
                matched_ids.append(doc_id)
                
    if matched_ids:
        print(f"--- Token Keyword Match for '{user_query}' ({len(matched_ids)} match(es)) ---")
        matched_subset = matched_ids[:top_k]
        result_df = df[df['Activity ID'].isin(matched_subset)].copy()
        result_df['sort_key'] = result_df['Activity ID'].map(lambda x: matched_subset.index(x))
        return result_df.sort_values('sort_key').drop(columns=['sort_key'])

    # 2. SEMANTIC VECTOR FALLBACK
    print(f"--- Semantic Vector Fallback for '{user_query}' ---")
    
    # Optional: Automatically prefix famous French cols if missing in vector search
    query_vector_str = user_query
    if "tourmalet" in raw_query and "col" not in raw_query:
        query_vector_str = user_query.replace("Tourmalet", "Col du Tourmalet").replace("tourmalet", "Col du Tourmalet")
        
    results = collection.query(
        query_texts=[query_vector_str], 
        n_results=top_k,
        include=["documents", "distances"]
    )
    
    if results and results['ids'] and len(results['ids'][0]) > 0:
        returned_ids = [str(x) for x in results['ids'][0]]
        distances = results['distances'][0]
        
        filtered_ids = [
            doc_id for doc_id, dist in zip(returned_ids, distances) 
            if dist <= max_distance
        ]
        
        if not filtered_ids:
            print("No vector matches met the similarity threshold.")
            return pd.DataFrame()
            
        result_df = df[df['Activity ID'].isin(filtered_ids)].copy()
        result_df['sort_key'] = result_df['Activity ID'].map(lambda x: filtered_ids.index(x))
        return result_df.sort_values('sort_key').drop(columns=['sort_key'])
    
    return pd.DataFrame()

# Test executions
print(search_rides("Tourmalet"))
print(search_rides("Ride over the Tourmalet"))

--- Token Keyword Match for 'Tourmalet' (2 match(es)) ---
     Activity ID                   Date  \
393  16082376772   7 okt 2025, 22:27:45   
769  12424582339  16 sep 2024, 13:14:44   

                                         Activity Name      Activity Type  \
393                           TransPyrenees Race - 6/7           Fietsrit   
769  ROUVY - Col du Tourmalet from Luz Saint Sauveu...  Virtuele fietsrit   

                                           Description  Verstreken tijd  \
393  I got up after a bit longer sleep than usual a...            80215   
769                                                NaN             4258   

    Distance (km)  Max. hartslag  Vergelijkbare poging  Woon-werkverkeer  ...  \
393        304,29          132.0                 263.0             False  ...   
769         19,21          154.0                 110.0             False  ...   

     Herstel Met huisdier Wedstrijd  Lange loop  Voor een doel  Met kind  \
393      0.0          NaN       Na

What with the size limitation for the indexing of the text data?